In [1]:
import pandas as pd
from sklearn.model_selection import KFold
pd.set_option('display.max_rows', 500)

In [2]:
data_artificial = pd.read_csv("data/20210205_all_expt_data_no_duplicates_solvent_calcs.csv")


In [3]:
# several molecules have 0 in experimetnal absorption results
data_artificial = data_artificial[data_artificial["peakwavs_max"] != 0]

In [4]:
# convert nm to ev
def nm2ev(wv):
    if wv is not None:
        return 1239.8 / wv
    else:
        return wv

In [5]:
# check problematic pigments which have short SMILES strings and 
# have a strong difference between experimental and computed data
# it is highly likely they are artifacts of text mining

# slope and intercept are the parameters colaculated by Greenman et al.

# to remove systematic error of TD-DFT calculation
slope = 1.82
intercept = 226.8

data_short_smiles = data_artificial[data_artificial["smiles"].apply(len) < 15]
difference = abs(data_short_smiles["peakwavs_max"] - slope * data_short_smiles["wb97xd3_def2svpd_orca_vac"] + intercept)

# we set a threshhold for difference between experimental and predicted values
suspicious_SMILES_small_mols = data_short_smiles[difference.isna() | (difference > 100.)]["smiles"]
len(suspicious_SMILES_small_mols)

46

In [6]:
# Examine where calculated data is different more than 100 nm from the experiment
# In this case there are potentially molecules which are not recognized correctly
# add these data to previously selected suspicious smiles

difference = abs(data_artificial["peakwavs_max"] - slope * data_artificial["wb97xd3_def2svpd_orca_vac"] + intercept)
suspicious_SMILES_big_mols = data_artificial[difference > 100.]["smiles"]
len(suspicious_SMILES_big_mols)

1813

In [7]:
# delete these suspicious SMILES string
data_artificial = data_artificial[~data_artificial["smiles"].isin(suspicious_SMILES_big_mols)]
data_artificial = data_artificial[~data_artificial["smiles"].isin(suspicious_SMILES_small_mols)] 
len(data_artificial)

18888

In [8]:
# nm to eV conversion

data_artificial["peakwavs_max"] = data_artificial["peakwavs_max"].apply(nm2ev)

In [9]:
# a set of natural compounds with wPBEPP86 calculated vertical excitation energies

natural_compounds = pd.read_csv("data/pigments_wb97xd4_tda_solv.csv")

In [10]:
# wb97xd4 - tda demonstrates a systematic error to remove it use slope and intercept obtained from linear models
# this is necessary to obtain linear unbiased estimation of mean absolute errors

slope = 0.71839382
intercept = 0.3203683403913211

natural_compounds["mae"] = abs(intercept + slope * natural_compounds['wavelength_wb97xd_tda_solvent'] - natural_compounds['lambda_max'])

In [11]:
# absorption energies are here in electron-volts;
# remove outliers from natural compounds dataset

len(natural_compounds[natural_compounds["mae"] < 0.5])

595

In [12]:
natural_compounds = natural_compounds[natural_compounds["mae"] < 0.5]
# rename column 'lambda_max' to "peakwavs_max"
natural_compounds['peakwavs_max'] = natural_compounds['lambda_max']
del natural_compounds['lambda_max']

In [19]:
# create 5 CV folds for train - test splitting for both artificial and natural compounds
kf_artificial = KFold(n_splits = 5, random_state = 1234, shuffle=True)
kf_natural = KFold(n_splits = 5, random_state = 5678, shuffle=True)

# prepare indices for all folds
[(art_train1, art_test1), (art_train2, art_test2), (art_train3, art_test3), (art_train4, art_test4), (art_train5, art_test5)] = list(kf_artificial.split(data_artificial))
[(nat_train1, nat_test1), (nat_train2, nat_test2), (nat_train3, nat_test3), (nat_train4, nat_test4), (nat_train5, nat_test5)] = list(kf_natural.split(natural_compounds))

# apply indices to dfs
data_art_train1 = data_artificial.iloc[art_train1.tolist()]
data_nat_train1 = natural_compounds.iloc[nat_train1.tolist()]
data_art_test1 = data_artificial.iloc[art_test1.tolist()]
data_nat_test1 = natural_compounds.iloc[nat_test1.tolist()]
data_art_train2 = data_artificial.iloc[art_train2.tolist()]
data_nat_train2 = natural_compounds.iloc[nat_train2.tolist()]
data_art_test2 = data_artificial.iloc[art_test2.tolist()]
data_nat_test2 = natural_compounds.iloc[nat_test2.tolist()]
data_art_train3 = data_artificial.iloc[art_train3.tolist()]
data_nat_train3 = natural_compounds.iloc[nat_train3.tolist()]
data_art_test3 = data_artificial.iloc[art_test3.tolist()]
data_nat_test3 = natural_compounds.iloc[nat_test3.tolist()]
data_art_train4 = data_artificial.iloc[art_train4.tolist()]
data_nat_train4 = natural_compounds.iloc[nat_train4.tolist()]
data_art_test4 = data_artificial.iloc[art_test4.tolist()]
data_nat_test4 = natural_compounds.iloc[nat_test4.tolist()]
data_art_train5 = data_artificial.iloc[art_train5.tolist()]
data_nat_train5 = natural_compounds.iloc[nat_train5.tolist()]
data_art_test5 = data_artificial.iloc[art_test5.tolist()]
data_nat_test5 = natural_compounds.iloc[nat_test5.tolist()]

# add training label column for chemprop 2
data_art_train1["split"] = ["train"] * len(data_art_train1)
data_art_test1["split"] = ["val"] * len(data_art_test1)
data_nat_train1["split"] = ["train"] * len(data_nat_train1)
data_nat_test1["split"] = ["test"] * len(data_nat_test1)
data_art_train2["split"] = ["train"] * len(data_art_train2)
data_art_test2["split"] = ["val"] * len(data_art_test2)
data_nat_train2["split"] = ["train"] * len(data_nat_train2)
data_nat_test2["split"] = ["test"] * len(data_nat_test2)
data_art_train3["split"] = ["train"] * len(data_art_train3)
data_art_test3["split"] = ["val"] * len(data_art_test3)
data_nat_train3["split"] = ["train"] * len(data_nat_train3)
data_nat_test3["split"] = ["test"] * len(data_nat_test3)
data_art_train4["split"] = ["train"] * len(data_art_train4)
data_art_test4["split"] = ["val"] * len(data_art_test4)
data_nat_train4["split"] = ["train"] * len(data_nat_train4)
data_nat_test4["split"] = ["test"] * len(data_nat_test4)
data_art_train5["split"] = ["train"] * len(data_art_train5)
data_art_test5["split"] = ["val"] * len(data_art_test5)
data_nat_train5["split"] = ["train"] * len(data_nat_train5)
data_nat_test5["split"] = ["test"] * len(data_nat_test5)

# union train data from artificial and natural compounds
data_art_train1 = data_art_train1[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_test1 = data_art_test1[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_train1 = data_nat_train1[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_test1 = data_nat_test1[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_train2 = data_art_train2[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_test2 = data_art_test2[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_train2 = data_nat_train2[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_test2 = data_nat_test2[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_train3 = data_art_train3[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_test3 = data_art_test3[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_train3 = data_nat_train3[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_test3 = data_nat_test3[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_train4 = data_art_train4[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_test4 = data_art_test4[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_train4 = data_nat_train4[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_test4 = data_nat_test4[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_train5 = data_art_train5[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_art_test5 = data_art_test5[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_train5 = data_nat_train5[['smiles', 'solvent', 'peakwavs_max', 'split']]
data_nat_test5 = data_nat_test5[['smiles', 'solvent', 'peakwavs_max', 'split']]

# data for training for chemprop 2.0
data_all1 = pd.concat((data_art_train1, data_art_test1, data_nat_train1, data_nat_test1))
data_all2 = pd.concat((data_art_train2, data_art_test2, data_nat_train2, data_nat_test2))
data_all3 = pd.concat((data_art_train3, data_art_test3, data_nat_train3, data_nat_test3))
data_all4 = pd.concat((data_art_train4, data_art_test4, data_nat_train4, data_nat_test4))
data_all5 = pd.concat((data_art_train5, data_art_test5, data_nat_train5, data_nat_test5))

[  1  10  15  18  19  23  27  33  35  38  43  50  51  53  54  59  61  66
  81  84  90  94  98  99 101 110 112 122 124 126 137 144 148 155 158 165
 169 177 181 185 186 189 190 203 206 215 229 230 231 235 242 251 259 260
 265 266 279 293 295 299 303 304 307 315 321 323 334 337 347 352 367 391
 392 399 406 407 410 412 419 422 426 434 438 448 451 454 459 461 464 472
 473 486 495 497 502 503 504 511 513 514 516 518 519 520 524 537 541 547
 554 556 564 567 569 571 573 578 579 587 592] [  8   9  11  16  22  25  31  34  39  44  45  47  56  62  64  68  69  70
  73  75  83  86  88  93 104 109 119 123 131 140 145 146 147 152 156 173
 184 192 199 210 214 216 218 219 227 232 236 237 238 245 250 252 256 261
 267 270 271 272 274 278 280 286 287 290 300 310 324 327 331 343 344 354
 355 357 358 363 370 378 383 384 388 396 398 404 414 417 425 432 433 446
 455 458 465 469 470 475 481 485 491 493 508 512 526 529 533 539 540 549
 550 552 557 560 562 563 568 585 586 589 593]


/var/folders/y4/h369999d6pl8gy56ppcbf13c0000gp/T/ipykernel_22879/962215481.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_art_train1["split"] = ["train"] * len(data_art_train1)
/var/folders/y4/h369999d6pl8gy56ppcbf13c0000gp/T/ipykernel_22879/962215481.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_art_test1["split"] = ["val"] * len(data_art_test1)
/var/folders/y4/h369999d6pl8gy56ppcbf13c0000gp/T/ipykernel_22879/962215481.py:35: SettingWithCopyWarning: 
A value is trying to be set on a 

In [21]:
# save all training data in csv format 
data_art_test1.to_csv("data/test_artificial1.csv")
data_art_test2.to_csv("data/test_artificial2.csv")
data_art_test3.to_csv("data/test_artificial3.csv")
data_art_test4.to_csv("data/test_artificial4.csv")
data_art_test5.to_csv("data/test_artificial5.csv")

data_nat_test1.to_csv("data/test_natural1.csv")
data_nat_test2.to_csv("data/test_natural2.csv")
data_nat_test3.to_csv("data/test_natural3.csv")
data_nat_test4.to_csv("data/test_natural4.csv")
data_nat_test5.to_csv("data/test_natural5.csv")

data_all1.to_csv("data/data_all1.csv")
data_all2.to_csv("data/data_all2.csv")
data_all3.to_csv("data/data_all3.csv")
data_all4.to_csv("data/data_all4.csv")
data_all5.to_csv("data/data_all5.csv")
